# Ppto27 · 20 Silver (limpieza, homologación, flags) — PySpark
1. **Versión vigente:** la última recibida de cada plantilla (las cargas `REF_` son la versión enviada, para los flags C).
2. **Montos:** las 24 columnas de meses pasan a formato largo y se interpretan ("1.500,00", "(300)", "S/ 200"…); letras, fechas y fórmulas rotas se marcan.
3. **Textos:** cada valor distinto (país, CeCo, cuenta, proveedor…) se limpia y homologa una vez y se cruza por broadcast.
4. **Flags** por archivo, línea y celda → estado: OK · OK con alerta · Cuarentena · Rechazada.

Se reconstruye completo en cada corrida: siempre refleja la última versión de cada plantilla.

In [ ]:
%pip install openpyxl --quiet

In [ ]:
import sys
sys.path.append('/Volumes/catalogo_ppto/esquema_ppto/vol_administrado/ppto27/00_codigo')            # librerías ppto27_*.py subidas al Volume
import time
import ppto27_lib as L, ppto27_spark as SP

t = time.time()
estados, n_flags = SP.silver(spark)
print(f'{estados} · {n_flags:,} flags · {time.time() - t:.0f} s')

In [ ]:
display(spark.sql(f"""
SELECT rubro, pais_carpeta, estado_archivo, COUNT(*) AS plantillas, SUM(lineas) AS lineas, SUM(lineas_ok) AS lineas_ok,
       ROUND(SUM(p27_ok), 2) AS p27_ok, ROUND(SUM(p27_cuarentena), 2) AS p27_cuarentena
FROM {L.tabla('slv_archivos')} GROUP BY ALL ORDER BY rubro, pais_carpeta"""))
display(spark.sql(f"""
SELECT severidad, codigo_flag, COUNT(*) AS casos, COUNT(DISTINCT id_plantilla) AS plantillas
FROM {L.tabla('slv_flags')} GROUP BY ALL ORDER BY array_position(array('BLOQUEANTE','ERROR','ALERTA','INFO'), severidad), casos DESC"""))